# NF→FF seeded indexing

Demonstrates `--nf-result-dir` (gap #3) and `--grains-file` (gap #4): seed
the FF indexer/refiner with a list of grains so it doesn't index from
scratch.

Two paths:

  - `nf_result_dir` — for layer N, look up `GrainsLayer{N}.csv`
    automatically. Useful for stacked NF→FF runs.
  - `grains_file` — explicit single Grains.csv used for every layer.

Either way, `MinNrSpots=1` is also written into the layer's paramstest
so sparsely-observed seeds aren't rejected.


In [ ]:
from pathlib import Path

from midas_ff_pipeline import Pipeline, PipelineConfig
from midas_ff_pipeline.config import LayerSelection
from midas_ff_pipeline.seeding import (
    resolve_grains_file_for_layer,
    patch_params_with_grains,
)


## 1. Inputs — edit these

In [ ]:
PARAMS = Path('/path/to/Parameters.txt')
ZARR = Path('/path/to/data.MIDAS.zip')
RESULT_DIR = Path('./nf_seeded_results')

# Pick exactly one:
NF_RESULT_DIR = Path('/path/to/nf_results')   # contains GrainsLayer1.csv, GrainsLayer2.csv, …
GRAINS_FILE = None                              # or: Path('/path/to/Grains.csv')


## 2. Quick check — what will the resolver pick for each layer?

`resolve_grains_file_for_layer` is the same logic the pipeline applies
internally. Helpful when you want to confirm the right NF file is being
picked up before kicking off a long run.

In [ ]:
for layer_nr in (1, 2, 3, 4, 5):
    seed = resolve_grains_file_for_layer(
        layer_nr=layer_nr,
        grains_file=str(GRAINS_FILE) if GRAINS_FILE else None,
        nf_result_dir=str(NF_RESULT_DIR) if NF_RESULT_DIR else None,
    )
    print(f'layer {layer_nr}: seed={seed}')


## 3. Run the pipeline with NF seeding

Pass `nf_result_dir` (or `grains_file`) at construction time; the pipeline
handles the per-layer resolution + paramstest patch transparently.

In [ ]:
config = PipelineConfig(
    result_dir=str(RESULT_DIR),
    params_file=str(PARAMS),
    zarr_path=str(ZARR),
    n_cpus=8,
    device='cpu',
    dtype='float64',
    layer_selection=LayerSelection(start=1, end=3),
    nf_result_dir=str(NF_RESULT_DIR) if NF_RESULT_DIR else None,
    grains_file=str(GRAINS_FILE) if GRAINS_FILE else None,
)
pipe = Pipeline(config=config)
pipe.run()

for r in pipe.layer_results:
    print(f'layer {r.layer_nr}: {r.n_grains} grains, {r.total_duration_s():.1f}s')


## 4. Confirm the GrainsFile actually went into paramstest

After `transforms` (or `cross_det_merge` for multi-det) runs, the resolver
patches `paramstest.txt` so downstream `midas-fit-grain` can pick it up.

In [ ]:
for r in pipe.layer_results:
    pt = Path(r.layer_dir) / 'paramstest.txt'
    if not pt.exists():
        print(f'layer {r.layer_nr}: paramstest missing')
        continue
    seeded = [l for l in pt.read_text().splitlines() if l.startswith('GrainsFile')]
    minspots = [l for l in pt.read_text().splitlines() if l.startswith('MinNrSpots')]
    print(f'layer {r.layer_nr}: {seeded[0] if seeded else "(no GrainsFile)"} | '
          f'{minspots[0] if minspots else "(no MinNrSpots)"}')
